# 01 — Initial Exploration
Load the BPI Challenge 2020 Travel Permit event log and inspect its raw structure.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
from src.load_event_log import load_xes_log, convert_to_dataframe
from src.inspect_log import summarize_columns, summarize_cases, summarize_activities

DATA_PATH = Path('../data/raw/PermitLog.xes')
TABLES_OUT = Path('../outputs/tables')
TABLES_OUT.mkdir(parents=True, exist_ok=True)

## 1. Load the XES file

In [3]:
log = load_xes_log(DATA_PATH)
print(f'Log loaded: {type(log)}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7065/7065 [00:00<00:00, 8890.35it/s] 


Log loaded: <class 'pandas.core.frame.DataFrame'>


## 2. Convert to DataFrame

In [4]:
df = convert_to_dataframe(log)
print(f'Shape: {df.shape}')
df.head()

Shape: (86581, 173)


,id,org:resource,concept:name,time:timestamp,org:role,case:OrganizationalEntity,case:ProjectNumber,case:TaskNumber,case:dec_id_0,case:ActivityNumber,...,case:Cost Type_14,case:Cost Type_10,case:Cost Type_11,case:Cost Type_12,case:Task_5,case:Task_4,case:Task_9,case:Task_8,case:Task_7,case:Task_6
0,rv_travel permit 76455_6,STAFF MEMBER,Start trip,2016-10-05 00:00:00+00:00,EMPLOYEE,organizational unit 65458,UNKNOWN,UNKNOWN,declaration 76457,activity 46005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rv_travel permit 76455_7,STAFF MEMBER,End trip,2016-10-05 00:00:00+00:00,EMPLOYEE,organizational unit 65458,UNKNOWN,UNKNOWN,declaration 76457,activity 46005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,st_step 76459_0,STAFF MEMBER,Permit SUBMITTED by EMPLOYEE,2017-04-06 13:32:10+00:00,EMPLOYEE,organizational unit 65458,UNKNOWN,UNKNOWN,declaration 76457,activity 46005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,st_step 76460_0,STAFF MEMBER,Permit FINAL_APPROVED by SUPERVISOR,2017-04-06 13:32:28+00:00,SUPERVISOR,organizational unit 65458,UNKNOWN,UNKNOWN,declaration 76457,activity 46005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,st_step 76461_0,STAFF MEMBER,Declaration SUBMITTED by EMPLOYEE,2017-04-07 13:38:14+00:00,EMPLOYEE,organizational unit 65458,UNKNOWN,UNKNOWN,declaration 76457,activity 46005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. DataFrame info and dtypes

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86581 entries, 0 to 86580
Columns: 173 entries, id to case:Task_6
dtypes: bool(1), datetime64[ns, UTC](1), float64(18), object(153)
memory usage: 113.7+ MB


## 4. Column summary (nulls, dtypes, unique counts)

In [6]:
col_summary = summarize_columns(df)
col_summary

,dtype,non_null,null,null_pct,unique
id,object,86581,0,0.00,86581
org:resource,object,86581,0,0.00,2
concept:name,object,86581,0,0.00,51
time:timestamp,"datetime64[ns, UTC]",86581,0,0.00,62741
org:role,object,86581,0,0.00,8
...,...,...,...,...,...
case:Task_4,object,549,86032,99.37,10
case:Task_9,object,159,86422,99.82,2
case:Task_8,object,159,86422,99.82,2
case:Task_7,object,159,86422,99.82,2


In [7]:
col_summary.to_csv(TABLES_OUT / 'column_summary.csv')
print('Saved column_summary.csv')

Saved column_summary.csv


## 5. Case and event counts

In [8]:
case_stats = summarize_cases(df)
for k, v in case_stats.items():
    print(f'{k}: {v}')

n_cases: 7065
n_events: 86581
events_per_case_mean: 12.254918612880397
events_per_case_median: 11.0
events_per_case_min: 3
events_per_case_max: 90


## 6. Unique activities and resources

In [9]:
activity_col = 'concept:name'
resource_col = 'org:resource' if 'org:resource' in df.columns else None

print(f'Unique activities: {df[activity_col].nunique()}')
if resource_col:
    print(f'Unique resources: {df[resource_col].nunique()}')
print()
print('Activities:')
print(df[activity_col].unique())

Unique activities: 51
Unique resources: 2

Activities:
['Start trip' 'End trip' 'Permit SUBMITTED by EMPLOYEE'
 'Permit FINAL_APPROVED by SUPERVISOR' 'Declaration SUBMITTED by EMPLOYEE'
 'Declaration FINAL_APPROVED by SUPERVISOR' 'Request Payment'
 'Payment Handled' 'Permit SAVED by EMPLOYEE'
 'Permit APPROVED by SUPERVISOR' 'Permit FINAL_APPROVED by DIRECTOR'
 'Declaration APPROVED by PRE_APPROVER'
 'Declaration APPROVED by ADMINISTRATION'
 'Permit APPROVED by PRE_APPROVER' 'Declaration REJECTED by PRE_APPROVER'
 'Declaration REJECTED by EMPLOYEE' 'Declaration SAVED by EMPLOYEE'
 'Send Reminder' 'Request For Payment SUBMITTED by EMPLOYEE'
 'Request For Payment FINAL_APPROVED by SUPERVISOR'
 'Request For Payment REJECTED by MISSING' 'Permit REJECTED by MISSING'
 'Permit REJECTED by SUPERVISOR' 'Permit REJECTED by EMPLOYEE'
 'Request For Payment APPROVED by PRE_APPROVER'
 'Declaration REJECTED by MISSING' 'Declaration APPROVED by SUPERVISOR'
 'Declaration FINAL_APPROVED by DIRECTOR'
 'P

## 7. Date range

In [10]:
time_col = 'time:timestamp'
df[time_col] = pd.to_datetime(df[time_col], utc=True)
print(f'Earliest event: {df[time_col].min()}')
print(f'Latest event  : {df[time_col].max()}')
print(f'Date range    : {(df[time_col].max() - df[time_col].min()).days} days')

Earliest event: 2016-10-05 00:00:00+00:00
Latest event  : 2021-09-01 00:00:00+00:00
Date range    : 1792 days


## 8. Sample traces (first 3 cases)

In [11]:
case_col = 'case:concept:name'
sample_cases = df[case_col].unique()[:3]
for c in sample_cases:
    trace = df[df[case_col] == c][[case_col, time_col, activity_col]].sort_values(time_col)
    print(trace.to_string(index=False))
    print()

  case:concept:name            time:timestamp                             concept:name
travel permit 76455 2016-10-05 00:00:00+00:00                               Start trip
travel permit 76455 2016-10-05 00:00:00+00:00                                 End trip
travel permit 76455 2017-04-06 13:32:10+00:00             Permit SUBMITTED by EMPLOYEE
travel permit 76455 2017-04-06 13:32:28+00:00      Permit FINAL_APPROVED by SUPERVISOR
travel permit 76455 2017-04-07 13:38:14+00:00        Declaration SUBMITTED by EMPLOYEE
travel permit 76455 2017-04-07 13:40:17+00:00 Declaration FINAL_APPROVED by SUPERVISOR
travel permit 76455 2017-04-11 15:03:43+00:00                          Request Payment
travel permit 76455 2017-04-13 17:30:53+00:00                          Payment Handled

  case:concept:name            time:timestamp                             concept:name
travel permit 76665 2016-11-21 00:00:00+00:00                               Start trip
travel permit 76665 2016-12-22 00:00:00+00

## 9. Activity frequency table

In [12]:
act_freq = summarize_activities(df)
act_freq.to_csv(TABLES_OUT / 'activity_frequency.csv', index=False)
act_freq

,activity,count,pct
0,Declaration SUBMITTED by EMPLOYEE,7574,8.75
1,Payment Handled,7544,8.71
2,Request Payment,7541,8.71
3,Permit SUBMITTED by EMPLOYEE,7331,8.47
4,Start trip,7065,8.16
5,End trip,7065,8.16
6,Permit FINAL_APPROVED by SUPERVISOR,6300,7.28
7,Permit APPROVED by ADMINISTRATION,5715,6.60
8,Declaration FINAL_APPROVED by SUPERVISOR,5641,6.52
9,Declaration APPROVED by ADMINISTRATION,4782,5.52
